# 04 — Improved logistic regression, LightGBM, and blending

**Group 119 — Henrik Remmen (541747)**  
**Kaggle team:** `[119] SINTEF Databreach`

This notebook implements the next controlled experiments:

1. Create a logistic-regression baseline submission with the same 20 features as the LightGBM baseline.
2. Tune logistic regularisation and compare cumulative feature blocks using purged chronological validation.
3. Train an improved LightGBM with exactly the same selected features.
4. Blend only if the out-of-fold predictions are sufficiently different and the blend improves both mean and worst-fold micro ROC-AUC.

> The notebook is intentionally saved **unexecuted**. Run it yourself, one section at a time. Local chronological validation is the primary selection criterion; Kaggle public score is a secondary diagnostic.


## 1. Setup and data

Run the cells in order. The feature-building cells do not train models.

In [ ]:
from pathlib import Path
import re
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
SUBMISSION_DIR = ROOT / "submissions"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
HORIZON = 168
N_JOBS = 4

print("Project root:", ROOT.resolve())


In [ ]:
def read_time_csv(path, time_column="date"):
    df = pd.read_csv(path)
    df[time_column] = pd.to_datetime(df[time_column], utc=True)
    return df

price = read_time_csv(DATA_DIR/"kernel"/"Historical_day_ahead_price_2015_2025.csv")
inflow = read_time_csv(DATA_DIR/"kernel"/"Historical_inflow_1958_2025.csv")
volume = read_time_csv(DATA_DIR/"kernel"/"Historical_volume_2015_2024.csv")
water_value = read_time_csv(DATA_DIR/"kernel"/"Synthetic_water_value_2015_2024.csv")
min_flow = read_time_csv(DATA_DIR/"extended"/"Constraint_min_flow.csv")
min_volume = read_time_csv(DATA_DIR/"extended"/"Constraint_min_volume.csv")
target_df = read_time_csv(DATA_DIR/"kernel"/"Unit_commitment_decisions.csv", "starttime")
prediction_mapping = read_time_csv(ROOT/"prediction_mapping.csv", "starttime")
sample_submission = pd.read_csv(ROOT/"sample_submission.csv")

with open(DATA_DIR/"extended"/"Tokke_Vinje_topology.yaml") as stream:
    topology = yaml.safe_load(stream)

target_pattern = re.compile(r"^result_committed_(.+)_t(\d+)$")
target_columns = [c for c in target_df if target_pattern.match(c)]
units = list(dict.fromkeys(target_pattern.match(c).group(1) for c in target_columns))

assert len(units) == 14
assert len(target_columns) == len(units)*HORIZON
assert target_columns == list(sample_submission.columns[1:])
assert set(np.unique(target_df[target_columns].to_numpy())).issubset({0, 1})

target_cube = np.stack([
    target_df[[f"result_committed_{unit}_t{h}" for h in range(HORIZON)]]
    .to_numpy(dtype=np.int8)
    for unit in units
], axis=1)

train_cases = target_df[["Run No", "starttime"]].copy()
test_cases = prediction_mapping[["Run No", "starttime"]].copy()
all_cases = pd.concat([
    train_cases.assign(split="train"),
    test_cases.assign(split="test"),
], ignore_index=True)

N_TRAIN_CASES = len(train_cases)
N_TEST_CASES = len(test_cases)
N_ALL_CASES = len(all_cases)

print("Target cube:", target_cube.shape)
print(f"Cases: {N_TRAIN_CASES:,} train + {N_TEST_CASES:,} test")


## 2. Build feature blocks

The blocks are cumulative so that every comparison changes only one coherent part of the representation.

In [ ]:
def numeric_indexed(df):
    return df.set_index("date").select_dtypes(include=np.number)

all_starts = pd.DatetimeIndex(all_cases["starttime"])
horizon_hour = np.tile(np.arange(HORIZON), N_ALL_CASES)
target_time = pd.DatetimeIndex(
    all_starts.repeat(HORIZON)
    + pd.to_timedelta(horizon_hour, unit="h")
)

def hourly_matrix(series):
    values = series.reindex(target_time).to_numpy()
    matrix = values.reshape(N_ALL_CASES, HORIZON)
    if not np.isfinite(matrix).all():
        raise ValueError(
            f"Missing/non-finite hourly inputs: {(~np.isfinite(matrix)).sum()}"
        )
    return matrix

def repeat_case(values):
    values = np.asarray(values)
    if len(values) != N_ALL_CASES:
        raise ValueError("Case-level array has incorrect length")
    return np.repeat(values, HORIZON)

def cyclic(values, period):
    angle = 2*np.pi*np.asarray(values)/period
    return np.sin(angle), np.cos(angle)

def clipped_shift(matrix, offset):
    indices = np.clip(np.arange(HORIZON) + offset, 0, HORIZON-1)
    return matrix[:, indices]

def centred_rolling_mean(matrix, window):
    return (
        pd.DataFrame(matrix).T
        .rolling(window=window, min_periods=1, center=True)
        .mean().T.to_numpy()
    )

feature_data = {}
feature_group = {}

def add_feature(name, values, group, dtype=np.float32):
    values = np.asarray(values)
    expected = N_ALL_CASES*HORIZON
    if values.ndim != 1 or len(values) != expected:
        raise ValueError(f"{name}: expected {expected} values, got {values.shape}")
    feature_data[name] = values.astype(dtype)
    feature_group[name] = group


In [ ]:
# Calendar features
hour = target_time.hour.to_numpy()
weekday = target_time.dayofweek.to_numpy()
month = target_time.month.to_numpy()

for name, values, period in (
    ("hour", hour, 24),
    ("weekday", weekday, 7),
    ("month", month-1, 12),
):
    sine, cosine = cyclic(values, period)
    add_feature(f"{name}_sin", sine, "calendar")
    add_feature(f"{name}_cos", cosine, "calendar")
add_feature("horizon_hour", horizon_hour, "calendar")

# Price features
price_series = price.set_index("date")["Day-ahead price (EUR/MWh)"]
price_matrix = hourly_matrix(price_series)
price_week_mean = price_matrix.mean(axis=1)
price_week_std = price_matrix.std(axis=1)
price_rank = pd.DataFrame(price_matrix).rank(axis=1, pct=True).to_numpy()

add_feature("price", price_matrix.ravel(), "price")
add_feature("price_week_mean", repeat_case(price_week_mean), "price")
add_feature("price_week_std", repeat_case(price_week_std), "price")
add_feature("price_week_percentile", price_rank.ravel(), "price")
add_feature(
    "price_minus_week_mean",
    (price_matrix-price_week_mean[:, None]).ravel(),
    "price",
)
add_feature(
    "price_week_zscore",
    ((price_matrix-price_week_mean[:, None])/(price_week_std[:, None]+1e-6)).ravel(),
    "price_windows",
)
add_feature("price_top_10pct", (price_rank >= .90).ravel(), "price_windows")
add_feature("price_bottom_10pct", (price_rank <= .10).ravel(), "price_windows")

for offset in (1, 3, 6, 24):
    lag = clipped_shift(price_matrix, -offset)
    lead = clipped_shift(price_matrix, offset)
    add_feature(f"price_lag_{offset}h", lag.ravel(), "price_windows")
    add_feature(f"price_lead_{offset}h", lead.ravel(), "price_windows")
    add_feature(
        f"price_change_from_{offset}h",
        (price_matrix-lag).ravel(),
        "price_windows",
    )
    add_feature(
        f"price_change_to_next_{offset}h",
        (lead-price_matrix).ravel(),
        "price_windows",
    )

for window in (3, 6, 12, 24, 48):
    rolling = centred_rolling_mean(price_matrix, window)
    add_feature(f"price_mean_{window}h", rolling.ravel(), "price_windows")
    add_feature(
        f"price_minus_mean_{window}h",
        (price_matrix-rolling).ravel(),
        "price_windows",
    )


In [ ]:
# Reservoir volumes and water values
reservoirs = [c for c in volume.columns if c != "date"]
capacity = {
    name: values["max_vol"]
    for name, values in topology["model"]["reservoir"].items()
}

start_volume = numeric_indexed(volume).reindex(all_starts)[reservoirs]
start_water = numeric_indexed(water_value).reindex(all_starts)[reservoirs]
end_dates = all_starts + pd.Timedelta(days=7)
end_water = numeric_indexed(water_value).reindex(end_dates)[reservoirs]

assert not start_volume.isna().any().any()
assert not start_water.isna().any().any()
assert not end_water.isna().any().any()

for reservoir in reservoirs:
    fill = start_volume[reservoir].to_numpy()/capacity[reservoir]
    w_start = start_water[reservoir].to_numpy()
    w_end = end_water[reservoir].to_numpy()
    add_feature(f"fill__{reservoir}", repeat_case(fill), "hydrology")
    add_feature(
        f"water_value_end__{reservoir}",
        repeat_case(w_end),
        "water_margins",
    )
    add_feature(
        f"water_value_change__{reservoir}",
        repeat_case(w_end-w_start),
        "water_margins",
    )
    add_feature(
        f"price_margin__{reservoir}",
        (price_matrix-w_end[:, None]).ravel(),
        "water_margins",
    )

system_fill = (
    start_volume.sum(axis=1).to_numpy()
    / sum(capacity[r] for r in reservoirs)
)
water_value_end_mean = end_water.mean(axis=1).to_numpy()

add_feature(
    "system_filling_ratio",
    repeat_case(system_fill),
    "hydrology",
)
add_feature(
    "water_value_end_mean",
    repeat_case(water_value_end_mean),
    "water_margins",
)
add_feature(
    "price_minus_water_value_mean",
    (price_matrix-water_value_end_mean[:, None]).ravel(),
    "water_margins",
)
add_feature(
    "price_to_abs_water_value_ratio",
    (price_matrix/(np.abs(water_value_end_mean[:, None])+1.0)).ravel(),
    "water_margins",
)


In [ ]:
# Inflow and environmental constraints
inflow_values = numeric_indexed(inflow)
inflow_matrices = {}

for column in inflow_values:
    matrix = hourly_matrix(inflow_values[column])
    inflow_matrices[column] = matrix
    add_feature(f"inflow__{column}", matrix.ravel(), "hydrology")
    add_feature(
        f"inflow_week_mean__{column}",
        repeat_case(matrix.mean(axis=1)),
        "hydrology",
    )

total_inflow_matrix = np.sum(list(inflow_matrices.values()), axis=0)
add_feature("inflow_total", total_inflow_matrix.ravel(), "hydrology")
add_feature(
    "inflow_total_week_mean",
    repeat_case(total_inflow_matrix.mean(axis=1)),
    "hydrology",
)
add_feature(
    "inflow_total_week_std",
    repeat_case(total_inflow_matrix.std(axis=1)),
    "hydrology",
)
add_feature(
    "inflow_total_24h_mean",
    centred_rolling_mean(total_inflow_matrix, 24).ravel(),
    "hydrology",
)
add_feature(
    "inflow_total_cumulative_mean",
    (
        np.cumsum(total_inflow_matrix, axis=1)
        /(np.arange(HORIZON)[None, :]+1)
    ).ravel(),
    "hydrology",
)

flow_values = numeric_indexed(min_flow)
minimum_volume_values = numeric_indexed(min_volume)
flow_matrices = {}
minimum_volume_matrices = {}

for column in flow_values:
    matrix = hourly_matrix(flow_values[column])
    flow_matrices[column] = matrix
    add_feature(f"min_flow__{column}", matrix.ravel(), "constraints")

for column in minimum_volume_values:
    matrix = hourly_matrix(minimum_volume_values[column])
    minimum_volume_matrices[column] = matrix
    add_feature(f"min_volume__{column}", matrix.ravel(), "constraints")
    if column in reservoirs:
        initial = start_volume[column].to_numpy()[:, None]
        add_feature(
            f"min_volume_slack__{column}",
            ((initial-matrix)/capacity[column]).ravel(),
            "constraints",
        )

flow_active_count = np.sum([x > 0 for x in flow_matrices.values()], axis=0)
volume_active_count = np.sum(
    [x > 0 for x in minimum_volume_matrices.values()],
    axis=0,
)
min_flow_total_matrix = np.sum(list(flow_matrices.values()), axis=0)
min_volume_total_matrix = np.sum(
    list(minimum_volume_matrices.values()),
    axis=0,
)

add_feature("min_flow_active_count", flow_active_count.ravel(), "constraints")
add_feature(
    "min_volume_active_count",
    volume_active_count.ravel(),
    "constraints",
)
add_feature("min_flow_total", min_flow_total_matrix.ravel(), "constraints")
add_feature("min_volume_total", min_volume_total_matrix.ravel(), "constraints")

variable_flow_columns = [
    c for c in flow_values
    if 0 < (flow_values[c] > 0).mean() < 1
]
variable_volume_columns = [
    c for c in minimum_volume_values
    if 0 < (minimum_volume_values[c] > 0).mean() < 1
]
variable_active = np.zeros((N_ALL_CASES, HORIZON), dtype=bool)
for column in variable_flow_columns:
    variable_active |= flow_matrices[column] > 0
for column in variable_volume_columns:
    variable_active |= minimum_volume_matrices[column] > 0
add_feature(
    "variable_constraint_active",
    variable_active.ravel(),
    "constraints",
)


In [ ]:
# Selected interactions suggested by the EDA
margin = price_matrix-water_value_end_mean[:, None]
system_fill_matrix = system_fill[:, None]

add_feature(
    "margin_x_system_fill",
    (margin*system_fill_matrix).ravel(),
    "interactions",
)
add_feature(
    "price_rank_x_system_fill",
    (price_rank*system_fill_matrix).ravel(),
    "interactions",
)
add_feature(
    "margin_x_total_inflow",
    (margin*total_inflow_matrix).ravel(),
    "interactions",
)
add_feature(
    "price_rank_x_hour_sin",
    price_rank.ravel()*feature_data["hour_sin"],
    "interactions",
)
add_feature(
    "price_rank_x_hour_cos",
    price_rank.ravel()*feature_data["hour_cos"],
    "interactions",
)
add_feature(
    "margin_x_constraint",
    margin.ravel()*variable_active.ravel(),
    "interactions",
)

X_all = pd.DataFrame(feature_data)
constant_features = [
    c for c in X_all
    if X_all[c].nunique(dropna=False) <= 1
]
X_all = X_all.drop(columns=constant_features)

assert X_all.shape[0] == N_ALL_CASES*HORIZON
assert not X_all.isna().any().any()
assert np.isfinite(X_all.to_numpy()).all()

BASELINE_FEATURES = [
    "hour_sin", "hour_cos", "weekday_sin", "weekday_cos",
    "month_sin", "month_cos", "horizon_hour",
    "price", "price_week_mean", "price_week_std",
    "price_week_percentile", "price_minus_week_mean",
    "water_value_end_mean", "price_minus_water_value_mean",
    "system_filling_ratio", "inflow_total",
    "inflow_total_24h_mean", "min_flow_total",
    "min_volume_total", "variable_constraint_active",
]

def group_features(group):
    return [
        name for name, value in feature_group.items()
        if value == group and name in X_all.columns
    ]

def unique_existing(names):
    return list(dict.fromkeys(c for c in names if c in X_all.columns))

FEATURE_SETS = {
    "baseline": unique_existing(BASELINE_FEATURES),
    "baseline_plus_price": unique_existing(
        BASELINE_FEATURES
        + group_features("price_windows")
    ),
    "baseline_plus_price_water": unique_existing(
        BASELINE_FEATURES
        + group_features("price_windows")
        + group_features("water_margins")
    ),
    "baseline_plus_price_water_hydrology": unique_existing(
        BASELINE_FEATURES
        + group_features("price_windows")
        + group_features("water_margins")
        + group_features("hydrology")
    ),
    "full_engineered": unique_existing(
        BASELINE_FEATURES
        + group_features("price_windows")
        + group_features("water_margins")
        + group_features("hydrology")
        + group_features("constraints")
        + group_features("interactions")
    ),
}

feature_set_summary = pd.DataFrame([
    {"feature_set": name, "n_features": len(columns)}
    for name, columns in FEATURE_SETS.items()
])
print("Feature matrix:", X_all.shape)
print("Removed constant features:", len(constant_features))
display(feature_set_summary)


## 3. Purged chronological validation and helpers

Training data for each fold ends before any target hour in the validation year. The competition metric is calculated over the complete `cases × generators × 168 hours` cube.

In [ ]:
train_starts = pd.DatetimeIndex(train_cases["starttime"])

def make_purged_folds(validation_years=(2020, 2021, 2022)):
    folds = []
    for year in validation_years:
        validation_start = pd.Timestamp(f"{year}-01-01", tz="UTC")
        validation_end = pd.Timestamp(f"{year+1}-01-01", tz="UTC")

        validation_cases = np.flatnonzero(
            (train_starts >= validation_start)
            & (train_starts < validation_end)
        )
        training_cases = np.flatnonzero(
            train_starts + pd.Timedelta(hours=HORIZON-1)
            < validation_start
        )
        assert len(training_cases) and len(validation_cases)
        assert (
            train_starts[training_cases].max()
            + pd.Timedelta(hours=HORIZON-1)
            < train_starts[validation_cases].min()
        )
        folds.append({
            "year": year,
            "train_cases": training_cases,
            "validation_cases": validation_cases,
        })
    return folds

FOLDS = make_purged_folds()

def rows_for_cases(case_ids):
    case_ids = np.asarray(case_ids)
    return (
        case_ids[:, None]*HORIZON
        + np.arange(HORIZON)[None, :]
    ).ravel()

def micro_auc(y_true_cube, y_pred_cube):
    return roc_auc_score(
        y_true_cube.reshape(len(y_true_cube), -1),
        y_pred_cube.reshape(len(y_pred_cube), -1),
        average="micro",
    )

def per_unit_auc(y_true_cube, y_pred_cube):
    return {
        unit: roc_auc_score(
            y_true_cube[:, i, :].ravel(),
            y_pred_cube[:, i, :].ravel(),
        )
        for i, unit in enumerate(units)
    }

def make_submission(prediction_cube, filename):
    if prediction_cube.shape != (N_TEST_CASES, len(units), HORIZON):
        raise ValueError(f"Unexpected prediction shape: {prediction_cube.shape}")

    submission_data = {"Run No": test_cases["Run No"].to_numpy()}
    for unit_idx, unit in enumerate(units):
        for horizon in range(HORIZON):
            submission_data[f"result_committed_{unit}_t{horizon}"] = (
                prediction_cube[:, unit_idx, horizon]
            )

    submission = pd.DataFrame(submission_data)[sample_submission.columns]
    probabilities = submission.iloc[:, 1:].to_numpy()

    assert submission.shape == sample_submission.shape
    assert submission["Run No"].equals(sample_submission["Run No"])
    assert np.isfinite(probabilities).all()
    assert probabilities.min() >= 0 and probabilities.max() <= 1

    path = SUBMISSION_DIR/filename
    submission.to_csv(path, index=False)
    print("Saved:", path.resolve())
    print("Shape:", submission.shape)
    print("Prediction range:", probabilities.min(), probabilities.max())
    return submission

fold_summary = pd.DataFrame([{
    "validation_year": fold["year"],
    "train_cases": len(fold["train_cases"]),
    "validation_cases": len(fold["validation_cases"]),
    "last_training_target": (
        train_starts[fold["train_cases"]].max()
        + pd.Timedelta(hours=HORIZON-1)
    ),
    "first_validation_start": train_starts[fold["validation_cases"]].min(),
} for fold in FOLDS])
display(fold_summary)


## 4. Logistic baseline submission — run this first

This deliberately reproduces the 20-feature representation used for `lgbm_baseline.csv`. Upload `logreg_baseline.csv` to Kaggle and compare it with the LightGBM public score **0.82273**.

Suggested Kaggle description: `LogReg baseline – same 20 features as LGBM`.

In [ ]:
# TRAINING CELL A — 14 logistic models on all training cases
all_train_rows = rows_for_cases(np.arange(N_TRAIN_CASES))
all_test_rows = rows_for_cases(np.arange(N_TRAIN_CASES, N_ALL_CASES))

X_train_baseline = X_all.loc[all_train_rows, BASELINE_FEATURES]
X_test_baseline = X_all.loc[all_test_rows, BASELINE_FEATURES]
logreg_baseline_test_prediction = np.empty(
    (N_TEST_CASES, len(units), HORIZON),
    dtype=np.float32,
)

for unit_idx, unit in enumerate(units):
    model = Pipeline([
        ("scale", StandardScaler()),
        ("classifier", LogisticRegression(
            C=1.0,
            solver="lbfgs",
            max_iter=500,
            tol=1e-4,
            random_state=RANDOM_SEED,
        )),
    ])
    y_train = target_cube[:, unit_idx, :].reshape(-1)
    model.fit(X_train_baseline, y_train)
    prediction = model.predict_proba(X_test_baseline)[:, 1]
    logreg_baseline_test_prediction[:, unit_idx, :] = prediction.reshape(
        N_TEST_CASES,
        HORIZON,
    )
    print(f"Finished {unit}")

logreg_baseline_submission = make_submission(
    logreg_baseline_test_prediction,
    "logreg_baseline.csv",
)
display(logreg_baseline_submission.iloc[:3, :8])


After this cell finishes:

1. Upload `submissions/logreg_baseline.csv`.
2. Record the public score.
3. Continue below. Do not choose a final model from the public score alone.

## 5. Improved logistic regression

First tune only `C` while keeping the baseline features fixed. Then keep the best `C` fixed while adding feature blocks. A feature set is promoted only when mean and worst-fold AUC are acceptable.

In [ ]:
EXPERIMENTS = {}

def evaluate_logistic(feature_columns, C, name):
    started_all = time.perf_counter()
    oof = np.full(target_cube.shape, np.nan, dtype=np.float32)
    fold_rows = []
    unit_rows = []

    for fold in FOLDS:
        started = time.perf_counter()
        train_case_ids = fold["train_cases"]
        val_case_ids = fold["validation_cases"]
        train_rows = rows_for_cases(train_case_ids)
        val_rows = rows_for_cases(val_case_ids)

        X_train = X_all.loc[train_rows, feature_columns]
        X_val = X_all.loc[val_rows, feature_columns]
        fold_prediction = np.empty(
            (len(val_case_ids), len(units), HORIZON),
            dtype=np.float32,
        )

        for unit_idx, unit in enumerate(units):
            y_train = target_cube[train_case_ids, unit_idx, :].reshape(-1)
            model = Pipeline([
                ("scale", StandardScaler()),
                ("classifier", LogisticRegression(
                    C=C,
                    solver="lbfgs",
                    max_iter=500,
                    tol=1e-4,
                    random_state=RANDOM_SEED,
                )),
            ])
            model.fit(X_train, y_train)
            prediction = model.predict_proba(X_val)[:, 1]
            fold_prediction[:, unit_idx, :] = prediction.reshape(
                len(val_case_ids),
                HORIZON,
            )

        oof[val_case_ids] = fold_prediction
        score = micro_auc(target_cube[val_case_ids], fold_prediction)
        fold_rows.append({
            "model": name,
            "validation_year": fold["year"],
            "micro_auc": score,
            "seconds": time.perf_counter()-started,
        })
        for unit, unit_score in per_unit_auc(
            target_cube[val_case_ids],
            fold_prediction,
        ).items():
            unit_rows.append({
                "model": name,
                "validation_year": fold["year"],
                "unit": unit,
                "auc": unit_score,
            })
        print(f"{name:42s} | {fold['year']} | AUC={score:.6f}")

    fold_scores = pd.DataFrame(fold_rows)
    result = {
        "name": name,
        "kind": "logistic",
        "C": C,
        "features": list(feature_columns),
        "oof": oof,
        "fold_scores": fold_scores,
        "per_unit_scores": pd.DataFrame(unit_rows),
        "mean_auc": fold_scores["micro_auc"].mean(),
        "worst_auc": fold_scores["micro_auc"].min(),
        "total_seconds": time.perf_counter()-started_all,
    }
    EXPERIMENTS[name] = result
    return result

def experiment_table():
    rows = []
    for result in EXPERIMENTS.values():
        rows.append({
            "model": result["name"],
            "kind": result["kind"],
            "n_features": len(result["features"]),
            "C": result.get("C", np.nan),
            "mean_auc": result["mean_auc"],
            "worst_auc": result["worst_auc"],
            "total_seconds": result["total_seconds"],
        })
    return (
        pd.DataFrame(rows)
        .sort_values(["mean_auc", "worst_auc"], ascending=False)
        .reset_index(drop=True)
    )


In [ ]:
# TRAINING CELL B — tune regularisation on baseline features
C_GRID = [0.03, 0.1, 0.3, 1.0, 3.0]
c_results = []

for C in C_GRID:
    result = evaluate_logistic(
        BASELINE_FEATURES,
        C=C,
        name=f"LogReg baseline C={C:g}",
    )
    c_results.append(result)

best_c_result = max(
    c_results,
    key=lambda result: (result["mean_auc"], result["worst_auc"]),
)
BEST_C = best_c_result["C"]

print("Selected C:", BEST_C)
display(experiment_table().style.format({
    "C": "{:.3g}",
    "mean_auc": "{:.6f}",
    "worst_auc": "{:.6f}",
    "total_seconds": "{:.1f}",
}))


In [ ]:
# TRAINING CELL C — compare cumulative feature blocks with fixed C
feature_results = []

for feature_set_name, columns in FEATURE_SETS.items():
    name = f"LogReg {feature_set_name} C={BEST_C:g}"
    if name in EXPERIMENTS:
        result = EXPERIMENTS[name]
    else:
        result = evaluate_logistic(columns, C=BEST_C, name=name)
    feature_results.append(result)

best_logistic_result = max(
    feature_results,
    key=lambda result: (result["mean_auc"], result["worst_auc"]),
)
SELECTED_FEATURE_SET = next(
    set_name
    for set_name, columns in FEATURE_SETS.items()
    if columns == best_logistic_result["features"]
)
SELECTED_FEATURES = FEATURE_SETS[SELECTED_FEATURE_SET]
BEST_LOGISTIC_NAME = best_logistic_result["name"]

print("Selected feature set:", SELECTED_FEATURE_SET)
print("Selected features:", len(SELECTED_FEATURES))
print("Best logistic experiment:", BEST_LOGISTIC_NAME)
display(experiment_table().style.format({
    "C": "{:.3g}",
    "mean_auc": "{:.6f}",
    "worst_auc": "{:.6f}",
    "total_seconds": "{:.1f}",
}))


## 6. Improved LightGBM with the same features

This isolates model class from feature quality: LightGBM gets exactly `SELECTED_FEATURES`. The import is delayed until here.

In [ ]:
try:
    import lightgbm as lgb
except OSError as error:
    if "libomp" in str(error).lower():
        raise OSError(
            "LightGBM needs OpenMP on macOS. Run 'brew install libomp', "
            "restart the Jupyter kernel, then rerun this cell."
        ) from error
    raise

def make_lgbm(n_estimators=1500):
    return lgb.LGBMClassifier(
        objective="binary",
        n_estimators=n_estimators,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=100,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.0,
        reg_lambda=2.0,
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS,
        force_col_wise=True,
        verbosity=-1,
    )

def evaluate_lgbm(feature_columns, name):
    started_all = time.perf_counter()
    oof = np.full(target_cube.shape, np.nan, dtype=np.float32)
    fold_rows = []
    unit_rows = []
    best_iterations = {unit: [] for unit in units}

    for fold in FOLDS:
        started = time.perf_counter()
        train_case_ids = fold["train_cases"]
        val_case_ids = fold["validation_cases"]
        train_rows = rows_for_cases(train_case_ids)
        val_rows = rows_for_cases(val_case_ids)

        X_train = X_all.loc[train_rows, feature_columns]
        X_val = X_all.loc[val_rows, feature_columns]
        fold_prediction = np.empty(
            (len(val_case_ids), len(units), HORIZON),
            dtype=np.float32,
        )

        for unit_idx, unit in enumerate(units):
            y_train = target_cube[train_case_ids, unit_idx, :].reshape(-1)
            y_val = target_cube[val_case_ids, unit_idx, :].reshape(-1)

            model = make_lgbm()
            model.fit(
                X_train,
                y_train,
                eval_set=[(X_val, y_val)],
                eval_metric="auc",
                callbacks=[lgb.early_stopping(100, verbose=False)],
            )
            prediction = model.predict_proba(
                X_val,
                num_iteration=model.best_iteration_,
            )[:, 1]
            fold_prediction[:, unit_idx, :] = prediction.reshape(
                len(val_case_ids),
                HORIZON,
            )
            best_iterations[unit].append(model.best_iteration_)

        oof[val_case_ids] = fold_prediction
        score = micro_auc(target_cube[val_case_ids], fold_prediction)
        fold_rows.append({
            "model": name,
            "validation_year": fold["year"],
            "micro_auc": score,
            "seconds": time.perf_counter()-started,
        })
        for unit, unit_score in per_unit_auc(
            target_cube[val_case_ids],
            fold_prediction,
        ).items():
            unit_rows.append({
                "model": name,
                "validation_year": fold["year"],
                "unit": unit,
                "auc": unit_score,
            })
        print(f"{name:42s} | {fold['year']} | AUC={score:.6f}")

    fold_scores = pd.DataFrame(fold_rows)
    result = {
        "name": name,
        "kind": "lightgbm",
        "features": list(feature_columns),
        "oof": oof,
        "fold_scores": fold_scores,
        "per_unit_scores": pd.DataFrame(unit_rows),
        "best_iterations": best_iterations,
        "mean_auc": fold_scores["micro_auc"].mean(),
        "worst_auc": fold_scores["micro_auc"].min(),
        "total_seconds": time.perf_counter()-started_all,
    }
    EXPERIMENTS[name] = result
    return result


In [ ]:
# TRAINING CELL D — improved LightGBM
IMPROVED_LGBM_NAME = f"LightGBM {SELECTED_FEATURE_SET}"
improved_lgbm_result = evaluate_lgbm(
    SELECTED_FEATURES,
    name=IMPROVED_LGBM_NAME,
)

display(experiment_table().style.format({
    "C": "{:.3g}",
    "mean_auc": "{:.6f}",
    "worst_auc": "{:.6f}",
    "total_seconds": "{:.1f}",
}))


## 7. Test whether blending is justified

The weight is selected entirely from chronological out-of-fold predictions. A weight of 1.0 means logistic only; 0.0 means LightGBM only. Prefer a blend only if it improves validation robustly, not merely because two models exist.

In [ ]:
# ANALYSIS CELL — fast; requires training cells C and D
logistic_oof = EXPERIMENTS[BEST_LOGISTIC_NAME]["oof"]
lgbm_oof = EXPERIMENTS[IMPROVED_LGBM_NAME]["oof"]

validation_case_ids = np.concatenate([
    fold["validation_cases"] for fold in FOLDS
])
logistic_errors = (
    target_cube[validation_case_ids].ravel()
    - logistic_oof[validation_case_ids].ravel()
)
lgbm_errors = (
    target_cube[validation_case_ids].ravel()
    - lgbm_oof[validation_case_ids].ravel()
)
error_correlation = np.corrcoef(logistic_errors, lgbm_errors)[0, 1]
prediction_correlation = np.corrcoef(
    logistic_oof[validation_case_ids].ravel(),
    lgbm_oof[validation_case_ids].ravel(),
)[0, 1]

blend_rows = []
for logistic_weight in np.linspace(0, 1, 21):
    fold_scores = []
    for fold in FOLDS:
        case_ids = fold["validation_cases"]
        blended = (
            logistic_weight*logistic_oof[case_ids]
            + (1-logistic_weight)*lgbm_oof[case_ids]
        )
        fold_scores.append(micro_auc(target_cube[case_ids], blended))

    blend_rows.append({
        "logistic_weight": logistic_weight,
        "lgbm_weight": 1-logistic_weight,
        "mean_auc": np.mean(fold_scores),
        "worst_auc": np.min(fold_scores),
        **{
            f"auc_{fold['year']}": score
            for fold, score in zip(FOLDS, fold_scores)
        },
    })

blend_results = (
    pd.DataFrame(blend_rows)
    .sort_values(["mean_auc", "worst_auc"], ascending=False)
    .reset_index(drop=True)
)
BEST_BLEND_WEIGHT = float(blend_results.loc[0, "logistic_weight"])

print(f"Prediction correlation: {prediction_correlation:.4f}")
print(f"Residual correlation:   {error_correlation:.4f}")
print(f"Selected logistic weight: {BEST_BLEND_WEIGHT:.2f}")
display(blend_results.head(10).style.format("{:.6f}"))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(
    blend_results.sort_values("logistic_weight")["logistic_weight"],
    blend_results.sort_values("logistic_weight")["mean_auc"],
    marker="o",
    label="Mean fold AUC",
)
ax.plot(
    blend_results.sort_values("logistic_weight")["logistic_weight"],
    blend_results.sort_values("logistic_weight")["worst_auc"],
    marker="o",
    label="Worst-fold AUC",
)
ax.axvline(BEST_BLEND_WEIGHT, color="black", linestyle="--", linewidth=1)
ax.set(
    title="OOF blend search",
    xlabel="Logistic-regression weight",
    ylabel="Micro ROC-AUC",
)
ax.legend()
plt.show()


## 8. Train selected models on all training data and create submissions

Run this only after the OOF comparisons above. It writes:

- `submissions/logreg_improved.csv`
- `submissions/lgbm_improved.csv`
- `submissions/blend_logreg_lgbm.csv`

The blend uses the OOF-selected weight. Upload the individual improved models before the blend so the leaderboard results remain interpretable.

In [ ]:
# TRAINING CELL E — final full-data models and three submission files
all_train_rows = rows_for_cases(np.arange(N_TRAIN_CASES))
all_test_rows = rows_for_cases(np.arange(N_TRAIN_CASES, N_ALL_CASES))
X_train_full = X_all.loc[all_train_rows, SELECTED_FEATURES]
X_test_full = X_all.loc[all_test_rows, SELECTED_FEATURES]

logreg_test_prediction = np.empty(
    (N_TEST_CASES, len(units), HORIZON),
    dtype=np.float32,
)
lgbm_test_prediction = np.empty_like(logreg_test_prediction)

for unit_idx, unit in enumerate(units):
    y_train = target_cube[:, unit_idx, :].reshape(-1)

    logistic_model = Pipeline([
        ("scale", StandardScaler()),
        ("classifier", LogisticRegression(
            C=BEST_C,
            solver="lbfgs",
            max_iter=500,
            tol=1e-4,
            random_state=RANDOM_SEED,
        )),
    ])
    logistic_model.fit(X_train_full, y_train)
    logistic_prediction = logistic_model.predict_proba(X_test_full)[:, 1]
    logreg_test_prediction[:, unit_idx, :] = logistic_prediction.reshape(
        N_TEST_CASES,
        HORIZON,
    )

    iterations = improved_lgbm_result["best_iterations"][unit]
    final_n_estimators = max(50, int(np.median(iterations)))
    lgbm_model = make_lgbm(n_estimators=final_n_estimators)
    lgbm_model.fit(X_train_full, y_train)
    lgbm_prediction = lgbm_model.predict_proba(X_test_full)[:, 1]
    lgbm_test_prediction[:, unit_idx, :] = lgbm_prediction.reshape(
        N_TEST_CASES,
        HORIZON,
    )

    print(f"{unit:18s}: logistic done, LightGBM trees={final_n_estimators}")

blend_test_prediction = (
    BEST_BLEND_WEIGHT*logreg_test_prediction
    + (1-BEST_BLEND_WEIGHT)*lgbm_test_prediction
)

logreg_improved_submission = make_submission(
    logreg_test_prediction,
    "logreg_improved.csv",
)
lgbm_improved_submission = make_submission(
    lgbm_test_prediction,
    "lgbm_improved.csv",
)
blend_submission = make_submission(
    blend_test_prediction,
    "blend_logreg_lgbm.csv",
)


## 9. What to do after running

1. Upload `logreg_baseline.csv` and record its public score next to the existing LightGBM baseline score **0.82273**.
2. Inspect the chronological table. The main decision numbers are mean AUC and worst-fold AUC.
3. Upload `logreg_improved.csv` and `lgbm_improved.csv`.
4. Upload the blend only if its OOF mean improves and its worst fold does not materially deteriorate.
5. Add all local and public results to `submissions/experiment_log.csv`.
6. Do not select the competition's final submissions until the strongest models have been tested. The private leaderboard can rank models differently from the public leaderboard.

A sensible promotion rule is:

`promote = mean_auc improves AND worst_fold_auc is not worse by more than 0.002`

If the blend's selected logistic weight is exactly 0 or 1, the OOF result says not to blend.


In [ ]:
# OOF EXPORT CELL — no fitting; run after training cells C and D
artifact_dir = ROOT / "artifacts" / "predictions"
artifact_dir.mkdir(parents=True, exist_ok=True)

if "best_logistic_result" not in globals():
    raise RuntimeError("Run training cell C before exporting logistic OOF.")
if "improved_lgbm_result" not in globals():
    raise RuntimeError("Run training cell D before exporting LightGBM OOF.")

np.savez_compressed(
    artifact_dir / "logreg_improved_oof.npz",
    oof=best_logistic_result["oof"],
)
np.savez_compressed(
    artifact_dir / "lgbm_improved_oof.npz",
    oof=improved_lgbm_result["oof"],
)

print("Saved:", artifact_dir / "logreg_improved_oof.npz")
print("Saved:", artifact_dir / "lgbm_improved_oof.npz")
